DATA LOADING

In [68]:
import pandas as pd
import numpy as np


In [69]:
df = pd.read_csv(r'C:\code\py\Gdc project\dirty_cafe_sales.csv')

DATA EXPLORATION

In [70]:
df.shape

(10000, 8)

In [71]:
df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


In [72]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [73]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [74]:
df.isnull().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

DATA CLEANING

In [75]:
df['Item'].unique()

<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',  'UNKNOWN',
 'Sandwich',        nan,    'ERROR',    'Juice',      'Tea']
Length: 11, dtype: str

In [76]:
df['Item'] = df['Item'].str.strip().str.title()

In [77]:
df[['Item', 'Price Per Unit']].head(15)

,Item,Price Per Unit
0,Coffee,2.0
1,Cake,3.0
2,Cookie,1.0
3,Salad,5.0
4,Coffee,2.0
5,Smoothie,4.0
6,Unknown,3.0
7,Sandwich,4.0
8,NaN,3.0
9,Sandwich,4.0


In [78]:
df.groupby('Item')['Price Per Unit'].unique()

Item
Cake                               [3.0, nan, UNKNOWN, ERROR]
Coffee                             [2.0, nan, ERROR, UNKNOWN]
Cookie                             [1.0, UNKNOWN, nan, ERROR]
Error       [1.5, 3.0, 5.0, nan, 4.0, 2.0, 1.0, UNKNOWN, E...
Juice                              [3.0, nan, UNKNOWN, ERROR]
Salad                              [5.0, ERROR, UNKNOWN, nan]
Sandwich                           [4.0, nan, ERROR, UNKNOWN]
Smoothie                           [4.0, nan, UNKNOWN, ERROR]
Tea                                [1.5, nan, ERROR, UNKNOWN]
Unknown     [3.0, 1.0, 5.0, 4.0, 1.5, 2.0, nan, UNKNOWN, E...
Name: Price Per Unit, dtype: object

In [79]:
df['Item'] = df['Item'].replace( ['Unknown', 'Error'], np.nan)

In [80]:
df['Price Per Unit'] = pd.to_numeric( df['Price Per Unit'], errors='coerce')

In [81]:
df.groupby('Item')['Price Per Unit'].unique()

Item
Cake        [3.0, nan]
Coffee      [2.0, nan]
Cookie      [1.0, nan]
Juice       [3.0, nan]
Salad       [5.0, nan]
Sandwich    [4.0, nan]
Smoothie    [4.0, nan]
Tea         [1.5, nan]
Name: Price Per Unit, dtype: object

In [82]:
df['Item'].isnull().sum()

np.int64(969)

In [83]:
df.loc[
    (df['Item'].isnull()) & (df['Price Per Unit'] == 1.0),'Item'] = 'Cookie'
df.loc[
    (df['Item'].isnull()) & (df['Price Per Unit'] == 1.5),'Item'] = 'Tea'

df.loc[(df['Item'].isnull()) &(df['Price Per Unit'] == 2.0),'Item'] = 'Coffee'

df.loc[(df['Item'].isnull()) &(df['Price Per Unit'] == 5.0),'Item'] = 'Salad'

In [84]:
df['Item'].value_counts(dropna=False)

Item
Coffee      1284
Salad       1270
Cookie      1209
Tea         1199
Juice       1171
Cake        1139
Sandwich    1131
Smoothie    1096
NaN          501
Name: count, dtype: int64

In [85]:
df[df['Item'].isnull()]['Price Per Unit'].value_counts(dropna=False)

Price Per Unit
3.0    234
4.0    213
NaN     54
Name: count, dtype: int64

In [86]:
df[(df['Price Per Unit'] == 4.0) | (df['Price Per Unit'] == 3.0)]['Item'].value_counts()

Item
Juice       1110
Cake        1085
Sandwich    1082
Smoothie    1036
Name: count, dtype: int64

In [87]:
df = df.dropna(subset=['Item'])

In [88]:
df['Item'].info()

<class 'pandas.Series'>
Index: 9499 entries, 0 to 9999
Series name: Item
Non-Null Count  Dtype
--------------  -----
9499 non-null   str  
dtypes: str(1)
memory usage: 148.4 KB


FIX QUANTITY

In [89]:
df['Quantity'].unique()

<StringArray>
['2', '4', '5', '3', '1', 'ERROR', 'UNKNOWN', nan]
Length: 8, dtype: str

In [90]:
df['Item'] = df['Item'].str.strip().str.title()

In [91]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')

In [92]:
df['Quantity'] = df['Quantity'].astype('Int64')

In [93]:
df['Quantity'].isnull().sum()

np.int64(452)

In [94]:
df['Quantity'].unique()

<IntegerArray>
[2, 4, 5, 3, 1, <NA>]
Length: 6, dtype: Int64

FIX PRICE PER UNIT

In [95]:
df['Price Per Unit'].unique()

array([2. , 3. , 1. , 5. , 4. , 1.5, nan])

In [96]:
df.groupby('Item')['Price Per Unit'].value_counts(dropna=False)

Item      Price Per Unit
Cake      3.0               1085
          NaN                 54
Coffee    2.0               1227
          NaN                 57
Cookie    1.0               1143
          NaN                 66
Juice     3.0               1110
          NaN                 61
Salad     5.0               1204
          NaN                 66
Sandwich  4.0               1082
          NaN                 49
Smoothie  4.0               1036
          NaN                 60
Tea       1.5               1133
          NaN                 66
Name: count, dtype: int64

In [97]:
df['Price Per Unit'] = df['Price Per Unit'].fillna(
    df.groupby('Item')['Price Per Unit'].transform('first')
)

In [98]:
df.groupby('Item')['Price Per Unit'].unique()

Item
Cake        [3.0]
Coffee      [2.0]
Cookie      [1.0]
Juice       [3.0]
Salad       [5.0]
Sandwich    [4.0]
Smoothie    [4.0]
Tea         [1.5]
Name: Price Per Unit, dtype: object

FIX TOTAL SPENT

In [99]:
df['Total Spent'].unique()

<StringArray>
[    '4.0',    '12.0',   'ERROR',    '10.0',    '20.0',    '16.0',    '25.0',
     '8.0',     '5.0',     '3.0',    '15.0',     '6.0',       nan, 'UNKNOWN',
     '2.0',     '9.0',     '1.0',     '7.5',     '4.5',     '1.5']
Length: 20, dtype: str

In [100]:
for col in ['Total Spent']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [101]:
for t in df['Total Spent'].unique():
    print(t)

4.0
12.0
nan
10.0
20.0
16.0
25.0
8.0
5.0
3.0
15.0
6.0
2.0
9.0
1.0
7.5
4.5
1.5


In [102]:
df['Total Spent'] = df['Total Spent'].fillna(
    df['Quantity'] * df['Price Per Unit']
)

In [103]:
df['Total Spent'].isnull().sum()

np.int64(20)

In [104]:
df[df['Total Spent'].isnull()][
    ['Quantity', 'Price Per Unit', 'Total Spent']
]

,Quantity,Price Per Unit,Total Spent
236,<NA>,5.0,<NA>
278,<NA>,3.0,<NA>
641,<NA>,3.0,<NA>
738,<NA>,4.0,<NA>
2796,<NA>,3.0,<NA>
3203,<NA>,4.0,<NA>
3224,<NA>,2.0,<NA>
3401,<NA>,1.5,<NA>
4257,<NA>,2.0,<NA>
5841,<NA>,1.0,<NA>


In [105]:
df.isnull().sum()

Transaction ID         0
Item                   0
Quantity             452
Price Per Unit         0
Total Spent           20
Payment Method      2453
Location            3089
Transaction Date     150
dtype: int64

FIX PAYMENT METHOD

In [106]:
df['Payment Method'].unique()

<StringArray>
['Credit Card', 'Cash', 'UNKNOWN', 'Digital Wallet', nan, 'ERROR']
Length: 6, dtype: str

In [107]:
df['Payment Method'] = df['Payment Method'].replace(['UNKNOWN', 'ERROR'], np.nan)

In [108]:
df['Payment Method'].isnull().sum()

np.int64(3022)

FIX LOCATION

In [109]:
df['Location'].unique()

<StringArray>
['Takeaway', 'In-store', 'UNKNOWN', nan, 'ERROR']
Length: 5, dtype: str

In [110]:
df['Location'] = df['Location'].replace(['UNKNOWN', 'ERROR'], np.nan)

In [111]:
df['Location'].isnull().sum()

np.int64(3759)

FIX DATE

In [112]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

In [113]:
df['Transaction Date'].isnull().sum()

np.int64(435)

In [114]:
df.sample()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
9008,TXN_2352308,Cookie,2,1.0,2.0,Cash,NaN,2023-11-10


In [115]:
df.info()

<class 'pandas.DataFrame'>
Index: 9499 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    9499 non-null   str           
 1   Item              9499 non-null   str           
 2   Quantity          9047 non-null   Int64         
 3   Price Per Unit    9499 non-null   float64       
 4   Total Spent       9479 non-null   Float64       
 5   Payment Method    6477 non-null   str           
 6   Location          5740 non-null   str           
 7   Transaction Date  9064 non-null   datetime64[us]
dtypes: Float64(1), Int64(1), datetime64[us](1), float64(1), str(4)
memory usage: 686.5 KB


FIX UPPERCASE

In [116]:
cols = ['Item', 'Payment Method', 'Location']

for col in cols:
    df[col] = df[col].str.lower()

In [117]:
df.sample(5)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
2205,TXN_2631671,cookie,1,1.0,1.0,credit card,NaN,NaT
1977,TXN_5836079,coffee,1,2.0,2.0,digital wallet,NaN,2023-07-14
3208,TXN_5828907,cookie,5,1.0,5.0,digital wallet,NaN,2023-05-13
5609,TXN_8876663,salad,<NA>,5.0,10.0,digital wallet,takeaway,2023-03-23
6497,TXN_1873800,smoothie,3,4.0,12.0,NaN,in-store,2023-02-01


HANDLING MISSING VALUES

In [118]:
df.isnull().sum()

Transaction ID         0
Item                   0
Quantity             452
Price Per Unit         0
Total Spent           20
Payment Method      3022
Location            3759
Transaction Date     435
dtype: int64

In [119]:
for col in ['Payment Method', 'Location']:
    df[col] = df[col].fillna(df[col].mode()[0])

In [120]:
df.isnull().sum()

Transaction ID        0
Item                  0
Quantity            452
Price Per Unit        0
Total Spent          20
Payment Method        0
Location              0
Transaction Date    435
dtype: int64

In [121]:
for col in ['Quantity', 'Total Spent']:
    df[col] = df[col].fillna(df[col].median())

In [122]:
df.isnull().sum()

Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    435
dtype: int64

In [123]:
df = df.dropna(subset=['Transaction Date'])

In [124]:
df.isnull().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

In [125]:
df.dtypes

Transaction ID                 str
Item                           str
Quantity                     Int64
Price Per Unit             float64
Total Spent                Float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object

SAVE CLEANED DATA

In [126]:
df.to_csv('cleaned_cafe-dataset.csv', index=False)

In [127]:
df.info()

<class 'pandas.DataFrame'>
Index: 9064 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    9064 non-null   str           
 1   Item              9064 non-null   str           
 2   Quantity          9064 non-null   Int64         
 3   Price Per Unit    9064 non-null   float64       
 4   Total Spent       9064 non-null   Float64       
 5   Payment Method    9064 non-null   str           
 6   Location          9064 non-null   str           
 7   Transaction Date  9064 non-null   datetime64[us]
dtypes: Float64(1), Int64(1), datetime64[us](1), float64(1), str(4)
memory usage: 655.0 KB
